# Maps of newly at-risk assets across voltage levels

## Import packages

In [ ]:
import time
from datetime import datetime
import os
import json
import yaml
import numpy as np 
import pandas as pd 
import pyarrow as pa
import glob
import xarray as xr
import random
import joblib
import math
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

import matplotlib as mpl
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]
mpl.rcParams['text.usetex'] = True

from src import input_ops
from src import df_ops
from src import file_ops

## Define Functions

In [ ]:
## --- Add MVA column ---
def mva_series_lines(df):
    if "NormAmps [A]" not in df.columns or "Nominal V [kV]" not in df.columns:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    amps = pd.to_numeric(df["NormAmps [A]"], errors="coerce").fillna(0.0)
    kv   = pd.to_numeric(df["Nominal V [kV]"], errors="coerce").fillna(0.0)
    return (amps * kv) / 1000.0

def mva_series_transformers(df):
    col = "kVA rating [kVA]" if "kVA rating [kVA]" in df.columns else None
    if col is None:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    kva = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    return kva / 1000.0

## --- Add secondary kV column ---
def secondary_kv_series_transformers(df):
    if "KV rating [kV]" not in df.columns:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    secondary_kv  = pd.to_numeric(df["KV rating [kV]"].str[-1], errors="coerce").fillna(0.0)
    return secondary_kv

# --- Helper functions to create table of overloaded count and capacity---
def below_threshold(column_data, threshold):
    return column_data < threshold
def in_threshold_range(column_data, threshold1, threshold2):
    return (column_data >= threshold1) & (column_data < threshold2)
def above_threshold(column_data, threshold):
    return column_data >= threshold


def risk_escalation_count(vmask):
    """Count number of assets with risk escalation under a voltage mask."""
    return (
        (vmask & mask_crit_F_not_B).sum()
        + (vmask & mask_cand_F_not_B).sum()
        + (vmask & mask_cand_B_to_crit_F).sum()
    )

def risk_escalation_mva(vmask):
    """Sum MVA of assets with risk escalation under a voltage mask."""
    return (
        df.loc[vmask & mask_crit_F_not_B, 'MVA rating [MVA]'].sum()
        + df.loc[vmask & mask_cand_F_not_B, 'MVA rating [MVA]'].sum()
        + df.loc[vmask & mask_cand_B_to_crit_F, 'MVA rating [MVA]'].sum()
    )

def add_percentage_columns_by_row_totals(
    df,
    totals_by_row,                 # list/array of length len(df), or dict keyed by 1-based row idx or by 'Metric' name
    start_col=1,                   # first numeric column index (B)
    n_numeric=4,                   # how many numeric columns to convert (B,F,F_minus_B,F_not_B)
    suffix="_pct",
    decimals=2,
):
    """
    Adds percentage columns for numeric columns [start_col : start_col+n_numeric).
    For row i, percentage = 100 * value / totals_by_row[i].

    totals_by_row can be:
      - list/np.array of length len(df) aligned to df row order, OR
      - dict keyed by 1-based row index (1..len(df)), OR
      - dict keyed by Metric names (df['Metric']).

    Notes:
      * If a row's total is 0/None/NaN, that row's percents become NaN (avoids div-by-zero).
      * Returns df with appended percentage columns.
    """
    num_cols = df.columns[start_col:start_col + n_numeric]

    # Build a Series aligned to df.index
    if isinstance(totals_by_row, (list, tuple, np.ndarray)):
        if len(totals_by_row) != len(df):
            raise ValueError(f"totals_by_row length {len(totals_by_row)} != number of rows {len(df)}")
        total_per_row = pd.Series(totals_by_row, index=df.index, dtype="float64")
    elif isinstance(totals_by_row, dict):
        total_per_row = pd.Series(np.nan, index=df.index, dtype="float64")
        if 'Metric' in df.columns:
            # try by Metric names first
            for i, m in df['Metric'].items():
                if m in totals_by_row:
                    total_per_row.loc[i] = totals_by_row[m]
        # fill remaining by 1-based row index if provided
        for i in df.index:
            idx1 = int(i - df.index.min() + 1)  # convert to 1-based
            if idx1 in totals_by_row and np.isnan(total_per_row.loc[i]):
                total_per_row.loc[i] = totals_by_row[idx1]
    else:
        raise TypeError("totals_by_row must be a list/array or dict")

    # Safe division
    totals_clean = total_per_row.replace({0: np.nan})
    pct = (
        df[num_cols].apply(pd.to_numeric, errors="coerce")
        .div(totals_clean, axis=0)
        .mul(100.0)
        .round(decimals)
    )
    pct.columns = [f"{c}{suffix}" for c in num_cols]
    return pd.concat([df, pct], axis=1)

def _assign_marker_size(df, size_bins, size_values):
    """
    Assigns marker sizes to transformers based on kVA rating bins.

    Parameters
    ----------
    df : DataFrame
        Must have column 'kVA rating [kVA]'.
    size_bins : list of tuple
        Each tuple is (lower_bound, upper_bound) in kVA.
    size_values : list of float
        Marker sizes (same length as size_bins).

    Returns
    -------
    Series with marker sizes.
    """
    kva = df['kVA rating [kVA]'].astype(float).to_numpy()
    marker_sizes = np.zeros_like(kva, dtype=float)

    for (low, high), size in zip(size_bins, size_values):
        mask = (kva >= low) & (kva < high)
        marker_sizes[mask] = size

    return pd.Series(marker_sizes, index=df.index)

def add_percentage_columns_by_row_ranges(
    df,
    total1=None, total1_rows=None,   # e.g., (1, 3)  -> rows 1..3 inclusive
    total2=None, total2_rows=None,   # e.g., (4, 6)  -> rows 4..6 inclusive
    start_col=1,                     # first numeric column index
    n_numeric=5,                     # how many numeric columns to convert
    suffix="_pct",
    decimals=2,
):
    """
    Adds percentage columns for numeric columns [start_col : start_col+n_numeric).
    percentage = 100 * value / row_total

    row_total is assigned by 1-based, inclusive row ranges:
      - rows in total1_rows -> total1
      - rows in total2_rows -> total2
      - all others -> NaN (percentages become NaN)

    Notes:
      * If a range is None or a total is None/0, that assignment is skipped.
      * If ranges overlap, later assignments take precedence.
    """
    num_cols = df.columns[start_col:start_col + n_numeric]
    nrows = len(df)
    pos = np.arange(1, nrows + 1)   # 1-based positions

    total_per_row = pd.Series(np.nan, index=df.index, dtype="float64")

    def apply_range(total, rng):
        if total is None or not rng or len(rng) != 2:
            return
        lo, hi = int(rng[0]), int(rng[1])
        if total == 0:
            return  # avoid div-by-zero (leave NaN)
        # clamp to valid bounds; inclusive
        lo = max(1, lo); hi = min(nrows, hi)
        if lo <= hi:
            mask = (pos >= lo) & (pos <= hi)
            total_per_row.iloc[mask] = float(total)

    apply_range(total1, total1_rows)
    apply_range(total2, total2_rows)

    pct = (
        df[num_cols].apply(pd.to_numeric, errors="coerce")
        .div(total_per_row, axis=0)
        .mul(100.0)
        .round(decimals)
    )
    pct.columns = [f"{c}{suffix}" for c in num_cols]
    return pd.concat([df, pct], axis=1)

    
def plot_donuts_and_maps_2x3_v3(
    transformers_overloaded,                
    transformers_summary_dict,              
    city_regions_to_run,                  
    smart_ds_year = '2018',
    level1_key=('2058', 'rcp45hotter'),
    cities=('AUS', 'GSO', 'SFO'),
    # ---------- donut config ----------
    outer_rows=('Cand+Crit LV #', 'Cand+Crit MV #', 'Cand+Crit HV #'),
    inner_rows=('Cand+Crit LV MVA', 'Cand+Crit MV MVA', 'Cand+Crit HV MVA'),
    outer_value_col='F_not_B',
    outer_percent_col='F_not_B_pct_of_all',
    inner_value_col='F_not_B',
    inner_percent_col='F_not_B_pct_of_all',
    donut_colors=('#50ad9f', '#e9c716', '#bc272d'),
    outer_radius=1.0,
    outer_width=0.2,
    inner_radius=0.62,
    inner_width=0.2,
    small_thresh_outer=80,
    small_thresh_inner=10,
    outer_unit='#',
    inner_unit=' MVA',
    font_title_city={'size': 12},
    font_ring_titles={'size': 10},
    font_labels={'size': 10},
    font_legend={'size': 10},
    outer_title='Future overloading \nby numbers',
    inner_title='Future \noverloading \nby capacity',
    outer_title_offset=0.18,
    city_label_offset=0.38,
    small_outer_separation=0.5,
    small_outer_slice_scale=7,
    value_decimals=0,
    percent_decimals=3,
    show_percent=True,
    # ---------- map config ----------
    hist_col='max_loading_historical_2018',
    fut_col='max_loading_rcp45hotter_2058',
    baseline_overload_thr=80,
    future_overload_thr=80,
    baseline_overload_thr_critical=100,
    future_overload_thr_critical=100,
    voltage_order=(0.12, 0.48, 12.47, 69.0),
    voltage_colors=None,  # dict {kV:"#hex"} or list of 4 hex; if None uses green-yellow-orange-red
    outline_color='#000000',
    outline_width_LV=0.9,
    outline_width_MV=0.9,
    outline_width_HV=0.9,
    fill_alpha=0.9,
    # ---------- global figure ----------
    figsize=(14, 9),
    suptitle=None,
    savepath = 'figures/maps/overloaded_xfers_by_kv/donuts_and_three_regions_GYR_v1.pdf'
):
    """
    Create a 2x3 figure:
      - Top row (a, b, c): maps of transformers (all regions per city),
        color by secondary kV, size by kVA, outlines for newly overloaded assets.
      - Bottom row (d, e, f): double-donut charts (one per city).

    V3 additions:
      - Adds a 10 km scale bar to each map using a local UTM projection.
      - Draws transformer markers from low to high voltage so higher-voltage
        markers appear above lower-voltage markers.
      - Draws all newly-overloaded black outlines in a separate final pass,
        so colored transformer markers can never cover them.
      - Supports separate outline widths for LV, MV, and HV transformers.

    Panel labels: (a)-(f) in top-left of each axis.
    """
    
    
    
    # Optional font overrides.
    # If None, text inherits its size from matplotlib rcParams.
    font_title_city = {} if font_title_city is None else font_title_city
    font_ring_titles = {} if font_ring_titles is None else font_ring_titles
    font_labels = {} if font_labels is None else font_labels

    # ======================================================================
    # Shared helpers
    # ======================================================================
    CITY_MAP = {'AUS': 'Austin', 'GSO': 'Greensboro', 'SFO': 'San Francisco'}

    # Local UTM projections (metres) for accurate physical distances / scale bars.
    CITY_UTM = {
        'AUS': 'EPSG:32614',  # Austin: UTM zone 14N
        'GSO': 'EPSG:32617',  # Greensboro: UTM zone 17N
        'SFO': 'EPSG:32610',  # San Francisco: UTM zone 10N
    }

    SCALE_BAR_KM = 10

    def _get_city_label(code):
        return CITY_MAP.get(code, code)

    # ----------------------------------------------------------------------
    # Donut helpers
    # ----------------------------------------------------------------------
    def _get_city_donut_df(city):
        try:
            return transformers_overloaded[level1_key][city]
        except Exception as e:
            raise KeyError(f"Missing donut data for {level1_key} / {city}: {e}")

    def _values_per_rows(df, rows, val_col, pct_col):
        sub = df.set_index('Metric')
        vals, pcts = [], []
        for r in rows:
            if r not in sub.index:
                vals.append(0.0)
                pcts.append(0.0)
            else:
                v = sub.at[r, val_col]
                p = sub.at[r, pct_col]
                v = float(v) if pd.notna(v) else 0.0
                p = float(p) if pd.notna(p) else 0.0
                vals.append(v)
                pcts.append(p)
        return np.array(vals, float), np.array(pcts, float)

    def _wedge_outer_radius(w, fallback_r):
        r = getattr(w, 'r', None)
        return r if r is not None else fallback_r

    def _wedge_width(w, default_width):
        width_attr = getattr(w, 'width', None)
        if width_attr is not None:
            return width_attr
        rin = getattr(w, '_inner_radius', None)
        r = getattr(w, 'r', None)
        if rin is not None and r is not None:
            return r - rin
        return default_width

    

    def _format_with_first_nonzero(x: float, base_decimals: int, max_decimals: int = 6) -> str:
        """
        Format a positive number with:
          - base_decimals decimals by default
          - BUT if that would round to 0 while x>0, increase decimals until the
            first non-zero digit is shown (up to max_decimals).

        Examples (base_decimals=1):
          8.423  -> "8.4"
          0.002  -> "0.002"
          0.0004 -> "0.0004"  (until max_decimals allows)
        """
        if not np.isfinite(x):
            return "0"
        if x == 0:
            return "0"

        sign = "-" if x < 0 else ""
        x_abs = abs(x)

        # Try base precision first
        d = max(0, int(base_decimals))
        s = f"{x_abs:.{d}f}"
        if float(s) != 0.0:
            # trim trailing zeros/dot for readability (optional)
            if d > 0:
                s = s.rstrip("0").rstrip(".")
            return sign + s

        # If base precision rounds to 0 but x>0, increase decimals until non-zero appears
        for d2 in range(d + 1, int(max_decimals) + 1):
            s2 = f"{x_abs:.{d2}f}"
            if float(s2) != 0.0:
                s2 = s2.rstrip("0").rstrip(".")
                return sign + s2

        # Fallback: scientific notation if still too small
        return sign + f"{x_abs:.{base_decimals}e}"


    def _two_line_label(val, pct, unit):
        # value formatting (UNCHANGED)
        if unit.strip() == '#' and value_decimals == 0:
            val_str = f"{int(round(val))}"
        else:
            val_fmt = f"{{:.{value_decimals}f}}"
            val_str = val_fmt.format(val)
            if value_decimals > 0:
                val_str = val_str.rstrip('0').rstrip('.')

        if not show_percent:
            return f"{val_str}{unit}"

        # ------------------------------------------------------------------
        # adaptive percent formatting
        # If pct is smaller than the resolution implied by percent_decimals,
        # increase decimals so the first non-zero digit is shown.
        # Example:
        #   percent_decimals = 1
        #   44.123  -> 44.1
        #   0.00123 -> 0.001
        # ------------------------------------------------------------------
        if pct == 0 or not np.isfinite(pct):
            pct_str = "0"
        else:
            # base decimals requested by user
            dec = percent_decimals

            # if value is too small, find decimals needed for first non-zero digit
            if pct < 10 ** (-percent_decimals):
                dec = max(
                    percent_decimals,
                    int(np.floor(-np.log10(abs(pct))))
                )

        pct_str = _format_with_first_nonzero(pct, base_decimals=percent_decimals, max_decimals=max(percent_decimals, 6))

        # ------------------------------------------------------------------

        return f"{val_str}{unit}\n({pct_str}\%)"

    
    
    def _annotate_wedges(
        ax, wedges, vals, pcts, unit,
        place_outside=False, small_thresh=None,
        ring_outer_radius=None, ring_width_default=None,
        inside_radius=None,
        for_outer=False, small_outer_separation=0.20
    ):
        small_thresh = small_thresh or 0.0

        for idx, (w, v, p) in enumerate(zip(wedges, vals, pcts)):
            if not np.isfinite(v) or v <= 0:
                continue

            txt = _two_line_label(v, p, unit)

            ang = (w.theta2 + w.theta1) / 2.0
            ang_rad = np.deg2rad(ang)
            r0 = _wedge_outer_radius(w, ring_outer_radius if ring_outer_radius is not None else 1.0)
            width = _wedge_width(w, ring_width_default if ring_width_default is not None else 0.3)
            
            
            r_mid = r0 - width / 6
            r_text = inside_radius if inside_radius is not None else r_mid
            x = r_text * np.cos(ang_rad)
            y = r_text * np.sin(ang_rad)

            if place_outside and v < small_thresh:
                r_out = r0 + 0.06
                xo = r_out * np.cos(ang_rad)
                yo = r_out * np.sin(ang_rad)

                if for_outer:
                    if idx == 1:   # MV
                        xo += small_outer_separation
                    elif idx == 2:  # HV
                        xo -= small_outer_separation

                ax.annotate(
                    txt, xy=(x + 0.02, y), xytext=(xo, yo), # Control location of arrow between label and outer ring
                    textcoords='data', ha='center', va='center',
                    arrowprops=dict(arrowstyle='-', lw=0.6, color='0.3'),
                    **font_labels
                )
            else:
                if (not place_outside) and v < small_thresh:
                    ax.annotate(
                        txt, xy=(x, y), xytext=(x * 0.85, y * 0.85),
                        textcoords='data', ha='center', va='center',
                        arrowprops=dict(arrowstyle='-', lw=0.6, color='0.3'),
                        **font_labels
                    )
                else:
                    ax.text(x, y, txt, ha='center', va='center', **font_labels)

    # ----------------------------------------------------------------------
    # Map helpers
    # ----------------------------------------------------------------------
    PERIOD_SUMMARY = level1_key # e.g., ('2058', 'rcp45hotter')

    def _get_city_region_df(city, region):
        return transformers_summary_dict[PERIOD_SUMMARY][(smart_ds_year, city, region)]

    def _add_scale_bar(ax, length_km=SCALE_BAR_KM, x_frac=0.08, y_frac=0.07):
        """Add a simple scale bar to a map whose CRS units are metres."""
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        x_range = xmax - xmin
        y_range = ymax - ymin
        length_m = float(length_km) * 1000.0

        # Position near the lower-left corner, while keeping the full bar inside the map.
        x0 = xmin + x_frac * x_range
        x0 = min(x0, xmax - length_m - 0.05 * x_range)
        y0 = ymin + y_frac * y_range
        x1 = x0 + length_m

        tick_h = 0.012 * y_range

        # White casing keeps the scale readable over the basemap.
        ax.plot([x0, x1], [y0, y0], color='white', lw=5.0,
                solid_capstyle='butt', zorder=100)
        ax.plot([x0, x0], [y0 - tick_h, y0 + tick_h], color='white', lw=5.0,
                solid_capstyle='butt', zorder=100)
        ax.plot([x1, x1], [y0 - tick_h, y0 + tick_h], color='white', lw=5.0,
                solid_capstyle='butt', zorder=100)

        ax.plot([x0, x1], [y0, y0], color='black', lw=2.0,
                solid_capstyle='butt', zorder=101)
        ax.plot([x0, x0], [y0 - tick_h, y0 + tick_h], color='black', lw=2.0,
                solid_capstyle='butt', zorder=101)
        ax.plot([x1, x1], [y0 - tick_h, y0 + tick_h], color='black', lw=2.0,
                solid_capstyle='butt', zorder=101)

        ax.text(
            (x0 + x1) / 2.0,
            y0 + 2.0 * tick_h,
            f'{length_km:g} km',
            ha='center', va='bottom', color='black', zorder=102,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=0.8)
        )

    # collect voltages across all city/regions so legend is globally consistent
    all_sec_v = []
    for city in cities:
        regs = city_regions_to_run[city]
        for region in regs:
            try:
                df = _get_city_region_df(city, region)
                sec = df['KV rating [kV]'].apply(
                    lambda x: (x[1] if isinstance(x, (list, tuple)) and len(x) > 1 else np.nan)
                )
                all_sec_v.append(sec.to_numpy(dtype=float))
            except Exception:
                continue
    all_sec_v = np.concatenate([v[np.isfinite(v)] for v in all_sec_v if v is not None])

    if voltage_order is None:
        unique_voltages = np.unique(all_sec_v)
        unique_voltages = np.sort(unique_voltages)
    else:
        present = set(all_sec_v.tolist())
        unique_voltages = [v for v in voltage_order if v in present]

    if len(unique_voltages) != 4:
        raise ValueError(
            f"Expected 4 unique secondary voltage values, got {len(unique_voltages)}: {unique_voltages}"
        )

    if isinstance(voltage_colors, dict):
        try:
            color_list = [mcolors.to_rgba(voltage_colors[v], alpha=fill_alpha) for v in unique_voltages]
        except KeyError as e:
            raise ValueError(f"Missing color for voltage {e.args[0]} in voltage_colors dict.")
    elif isinstance(voltage_colors, (list, tuple)):
        if len(voltage_colors) != 4:
            raise ValueError("voltage_colors list must have exactly 4 HEX colors.")
        color_list = [mcolors.to_rgba(c, alpha=fill_alpha) for c in voltage_colors]
    else:
        # default G-Y-O-R palette
        default_hex = ['#81FB48', '#FCDF90', '#F7A35C', '#EA848A']
        color_list = [mcolors.to_rgba(c, alpha=fill_alpha) for c in default_hex]

    cmap = ListedColormap(color_list)
    v_to_idx = {v: i for i, v in enumerate(unique_voltages)}

    # ======================================================================
    # Create 2x3 figure
    # ======================================================================
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    axes = np.atleast_2d(axes)

    if suptitle:
        fig.suptitle(suptitle, y=0.98)

    # Legend for LV/MV/HV colors (donuts) at top center
    legend_handles_donut = [
        # Patch(facecolor=donut_colors[0], edgecolor='none', label='Low-voltage'),
        # Patch(facecolor=donut_colors[1], edgecolor='none', label='Medium-voltage'),
        # Patch(facecolor=donut_colors[2], edgecolor='none', label='High-voltage'),
        
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=donut_colors[0],
            markeredgecolor='none',
            markeredgewidth=0,
            markersize=10,
            label='Low-voltage'
        ),
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=donut_colors[1],
            markeredgecolor='none',
            markeredgewidth=0,
            markersize=10,
            label='Medium-voltage'
        ),
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=donut_colors[2],
            markeredgecolor='none',
            markeredgewidth=0,
            markersize=10,
            label='High-voltage'
        ),
        
        # outline-only circle
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor='none',
            markeredgecolor='black',
            markeredgewidth=1.0,
            markersize=10,
            label='Shift to at-risk or critical loading'
        ),
    ]

    legend_kwargs = {}
    if font_legend is not None:
        legend_kwargs["prop"] = font_legend

    fig.legend(
        handles=legend_handles_donut,
        loc='upper center',
        ncol=4,
        bbox_to_anchor=(0.5, 1.05),
        frameon=False,
        **legend_kwargs
    )    
    
    # ======================================================================
    # Row 2: maps (d, e, f) — one map per city, all regions overlaid
    # ======================================================================
    for ax, city in zip(axes[0, :], cities):
        ax.set_aspect('equal')

        # gather all regions for this city
        regs = city_regions_to_run[city]
        dfs = []
        for region in regs:
            try:
                df_reg = _get_city_region_df(city, region)
                df_reg = df_reg.copy()
                df_reg['Region'] = region
                dfs.append(df_reg)
            except Exception:
                continue

        if not dfs:
            ax.set_axis_off()
            continue

        df_city = pd.concat(dfs, ignore_index=True)

        # marker size bins (same as your example)
        size_bins = [
            (0, 1e2), (1e2, 1e3), (1e3, 1e4),
            (1e4, 1e5), (1e5, 1e6), (1e6, 1e7), (1e7, np.inf)
        ]
        size_values = [5, 10, 20, 80, 100, 150, 300]
        df_city['Marker Size'] = _assign_marker_size(df_city, size_bins, size_values)

        # secondary voltage & class
        df_city['Secondary kV'] = df_city['KV rating [kV]'].apply(
            lambda x: (x[1] if isinstance(x, (list, tuple)) and len(x) > 1 else np.nan)
        ).astype(float)
        df_city['Voltage Class'] = df_city['Secondary kV'].map(v_to_idx)

        # newly overloaded = candidate or critical
        df_city['Newly Overloaded'] = (
            ((df_city[hist_col] < baseline_overload_thr) &
             (df_city[fut_col] >= future_overload_thr)) |
            ((df_city[hist_col] < baseline_overload_thr_critical) &
             (df_city[fut_col] >= future_overload_thr_critical))
        )

        if city not in CITY_UTM:
            raise ValueError(
                f"No local UTM CRS is defined for city '{city}'. "
                f"Add it to CITY_UTM inside plot_donuts_and_maps_2x3_v3()."
            )
        map_crs = CITY_UTM[city]

        gdf = gpd.GeoDataFrame(
            df_city,
            geometry=gpd.points_from_xy(df_city['Long'], df_city['Lat']),
            crs='EPSG:4326'
        ).to_crs(map_crs)

        # --------------------------------------------------------------
        # colored transformer markers
        # --------------------------------------------------------------

        for voltage_idx, voltage in enumerate(unique_voltages):
            gdf_v = gdf[gdf['Secondary kV'] == voltage].copy()
            if gdf_v.empty:
                continue

            gdf_v = gdf_v.sort_values('Marker Size', ascending=False)

            gdf_v.plot(
                ax=ax,
                color=color_list[voltage_idx],
                markersize=gdf_v['Marker Size'],
                marker='o',
                edgecolor='none',
                zorder=2 + voltage_idx
            )

        # --------------------------------------------------------------
        #  newly-overloaded outlines
        # --------------------------------------------------------------
        for voltage_idx, voltage in enumerate(unique_voltages):
            gdf_v_over = gdf[
                (gdf['Secondary kV'] == voltage) &
                (gdf['Newly Overloaded'])
            ].copy()

            if gdf_v_over.empty:
                continue

            gdf_v_over = gdf_v_over.sort_values('Marker Size', ascending=False)

            # Match the existing LV / MV / HV definitions:
            #   LV: secondary kV < 1
            #   MV: 1 <= secondary kV <= 35
            #   HV: secondary kV > 35
            if voltage < 1:
                this_outline_width = outline_width_LV
            elif voltage <= 35:
                this_outline_width = outline_width_MV
            else:
                this_outline_width = outline_width_HV

            gdf_v_over.plot(
                ax=ax,
                color='none',
                edgecolor=outline_color,
                markersize=gdf_v_over['Marker Size'] * 1.2,
                linewidth=this_outline_width,
                marker='o',
                zorder=50 + voltage_idx
            )

        # Contextily warps the basemap into the city's local UTM projection.
        cx.add_basemap(
            ax,
            source=cx.providers.CartoDB.Positron,
            crs=map_crs
        )
        for text in ax.texts:
            # shrink basemap attribution
            text.set_color('gray')
            text.set_alpha(0.5)

        # 10 km scale bar, using metre-based UTM coordinates.
        _add_scale_bar(ax)

        ax.set_axis_off()
        ax.set_title(f"{_get_city_label(city)}", pad=2)

    # Store the numerical values shown in the donut plots
    donut_records = []  
    
    # ======================================================================
    # Row 1: donuts (a, b, c)
    # ======================================================================
    for ax, city in zip(axes[1, :], cities):
        ax.set_aspect('equal')

        df = _get_city_donut_df(city)
        outer_vals, outer_pcts = _values_per_rows(df, outer_rows, outer_value_col, outer_percent_col)
        inner_vals, inner_pcts = _values_per_rows(df, inner_rows, inner_value_col, inner_percent_col)

        outer_vals = np.where(np.isfinite(outer_vals), outer_vals, 0.0)
        inner_vals = np.where(np.isfinite(inner_vals), inner_vals, 0.0)
        outer_vals = np.where(outer_vals < 0, 0.0, outer_vals)
        inner_vals = np.where(inner_vals < 0, 0.0, inner_vals)
        outer_pcts = np.where(np.isfinite(outer_pcts), outer_pcts, 0.0)
        inner_pcts = np.where(np.isfinite(inner_pcts), inner_pcts, 0.0)
        
        # Save the values shown in the donuts
        for voltage_level, number, number_pct, capacity, capacity_pct in zip(
            ('LV', 'MV', 'HV'),
            outer_vals,
            outer_pcts,
            inner_vals,
            inner_pcts,
        ):
            donut_records.append({
                'City': _get_city_label(city),
                'Voltage level': voltage_level,
                'Number': number,
                'Number [%]': number_pct,
                'Capacity [MVA]': capacity,
                'Capacity [%]': capacity_pct,
    })        

        # city label
        ax.text(
            0, outer_radius + city_label_offset,
            _get_city_label(city),
            ha='center', va='center', **font_title_city
        )

        # outer ring
        if np.allclose(outer_vals.sum(), 0):
            outer_vals_plot = np.array([1, 1, 1], float)
            colors_outer = ['#dddddd'] * 3
        else:
            outer_vals_plot = outer_vals.astype(float).copy()
            if small_outer_slice_scale != 1.0 and small_thresh_outer is not None:
                small_mask = (outer_vals_plot > 0) & (outer_vals_plot < small_thresh_outer)
                outer_vals_plot[small_mask] *= float(small_outer_slice_scale)
            colors_outer = donut_colors

        wedges_outer, _ = ax.pie(
            outer_vals_plot,
            radius=outer_radius,
            colors=colors_outer,
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=outer_width, edgecolor='white')
        )

        # outer title (bottom-left of the donut panel, in axes coordinates)
        ax.text(
            0.04, 0.08,                      # x,y in axes fraction (0..1)
            outer_title,
            transform=ax.transAxes,          # anchor to the axes box
            ha='left', va='bottom',
            **font_ring_titles
        )



        _annotate_wedges(
            ax, wedges_outer, outer_vals, outer_pcts, outer_unit,
            place_outside=True, small_thresh=small_thresh_outer,
            ring_outer_radius=outer_radius, ring_width_default=outer_width,
            for_outer=True, small_outer_separation=small_outer_separation
        )

        # inner ring
        if np.allclose(inner_vals.sum(), 0):
            inner_vals_plot = np.array([1, 1, 1], float)
            colors_inner = ['#eeeeee'] * 3
        else:
            inner_vals_plot = inner_vals
            colors_inner = donut_colors

        wedges_inner, _ = ax.pie(
            inner_vals_plot,
            radius=inner_radius,
            colors=colors_inner,
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=inner_width, edgecolor='white')
        )

        ax.text(0, 0, inner_title, ha='center', va='center', **font_ring_titles)

        inner_text_radius = inner_radius - inner_width * 0.4
        _annotate_wedges(
            ax, wedges_inner, inner_vals, inner_pcts, inner_unit,
            place_outside=False, small_thresh=small_thresh_inner,
            ring_outer_radius=inner_radius, ring_width_default=inner_width,
            inside_radius=inner_text_radius
        )

        ax.set_axis_off()


    # ======================================================================
    # Panel labels (a)-(f)
    # ======================================================================
    panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']
    k = 0
    for i in range(2):
        for j in range(3):
            ax = axes[i, j]
            ax.text(
                0.01, 0.99, panel_labels[k],
                transform=ax.transAxes,
                ha='left', va='top',
                fontweight='bold'
            )
            k += 1

    # plt.tight_layout(rect=(0, 0, 1, 0.98))
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath,bbox_inches='tight', pad_inches=0.5, dpi=600) # PDF forma
    plt.show()
    
    donut_df = pd.DataFrame(donut_records)
    return donut_df

def plot_donuts_and_maps_4x3_v1(
    transformers_overloaded,              
    transformers_summary_dict,             
    city_regions_to_run,                   
    smart_ds_year = '2018',
    level1_key=('2058', 'rcp45hotter'),
    cities=('AUS', 'GSO', 'SFO'),
    # ---------- donut config ----------
    outer_rows=('Cand+Crit LV #', 'Cand+Crit MV #', 'Cand+Crit HV #'),
    inner_rows=('Cand+Crit LV MVA', 'Cand+Crit MV MVA', 'Cand+Crit HV MVA'),
    outer_value_col='F_not_B',
    outer_percent_col='F_not_B_pct_of_all',
    inner_value_col='F_not_B',
    inner_percent_col='F_not_B_pct_of_all',
    donut_colors=('#50ad9f', '#e9c716', '#bc272d'),
    outer_radius=1.0,
    outer_width=0.2,
    inner_radius=0.62,
    inner_width=0.2,
    small_thresh_outer=80,
    small_thresh_inner=10,
    outer_unit='#',
    inner_unit=' MVA',
    font_title_city={'size': 12},
    font_ring_titles={'size': 10},
    font_labels={'size': 10},
    font_legend={'size': 10},
    outer_title='Future overloading \nby numbers',
    inner_title='Future \noverloading \nby capacity',
    outer_title_offset=0.18,
    city_label_offset=0.38,
    small_outer_separation=0.5,
    small_outer_slice_scale=7,
    value_decimals=0,
    percent_decimals=3,
    show_percent=True,
    # ---------- map config ----------
    hist_col='max_loading_historical_2018',
    fut_col='max_loading_rcp45hotter_2058',
    baseline_overload_thr=80,
    future_overload_thr=80,
    baseline_overload_thr_critical=100,
    future_overload_thr_critical=100,
    voltage_order=(0.12, 0.48, 12.47, 69.0),
    voltage_colors=None,  # dict {kV:"#hex"} or list of 4 hex; if None uses green-yellow-orange-red
    outline_color='#000000',
    outline_width_LV=0.9,
    outline_width_MV=0.9,
    outline_width_HV=0.9,
    fill_alpha=0.9,
    # ---------- global figure ----------
    figsize=(14, 15),
    suptitle=None,
    savepath = 'figures/maps/overloaded_xfers_by_kv/donuts_and_nine_regions_GYR_v1.pdf'
):
    """
    Create a 4x3 figure:
      - Rows 1--3: one transformer map per city-region combination (9 maps total).
        Columns correspond to cities; rows correspond to Region 1, Region 2,
        and Region 3 in the order supplied in city_regions_to_run[city].
      - Row 4: double-donut charts (one per city)
    Each map:
      - uses local UTM projection and 10 km scale bar;
      - draws transformer markers from low to high voltage;
      - draws all newly-overloaded black outlines after all colored markers;
      - supports separate outline widths for LV, MV, and HV transformers;
      - includes a small label such as ``Region 1 (P1U)``.

    Panel labels: (a)-(l) in the top-left of each axis.
    """
    

    def _font_kwargs(font_dict, name):
        if font_dict is None:
            return {}
        if not isinstance(font_dict, dict):
            raise TypeError(
                f"{name} must be None or a dictionary, e.g. {{'size': 13}}."
            )
        return font_dict

    font_title_city = _font_kwargs(font_title_city, "font_title_city")
    font_ring_titles = _font_kwargs(font_ring_titles, "font_ring_titles")
    font_labels = _font_kwargs(font_labels, "font_labels")

    # ======================================================================
    # Shared helpers
    # ======================================================================
    CITY_MAP = {'AUS': 'Austin', 'GSO': 'Greensboro', 'SFO': 'San Francisco'}

    # Local UTM projections (metres) for accurate physical distances / scale bars.
    CITY_UTM = {
        'AUS': 'EPSG:32614',  # Austin: UTM zone 14N
        'GSO': 'EPSG:32617',  # Greensboro: UTM zone 17N
        'SFO': 'EPSG:32610',  # San Francisco: UTM zone 10N
    }

    SCALE_BAR_KM = 10

    def _get_city_label(code):
        return CITY_MAP.get(code, code)

    # ----------------------------------------------------------------------
    # Donut helpers
    # ----------------------------------------------------------------------
    def _get_city_donut_df(city):
        try:
            return transformers_overloaded[level1_key][city]
        except Exception as e:
            raise KeyError(f"Missing donut data for {level1_key} / {city}: {e}")

    def _values_per_rows(df, rows, val_col, pct_col):
        sub = df.set_index('Metric')
        vals, pcts = [], []
        for r in rows:
            if r not in sub.index:
                vals.append(0.0)
                pcts.append(0.0)
            else:
                v = sub.at[r, val_col]
                p = sub.at[r, pct_col]
                v = float(v) if pd.notna(v) else 0.0
                p = float(p) if pd.notna(p) else 0.0
                vals.append(v)
                pcts.append(p)
        return np.array(vals, float), np.array(pcts, float)

    def _wedge_outer_radius(w, fallback_r):
        r = getattr(w, 'r', None)
        return r if r is not None else fallback_r

    def _wedge_width(w, default_width):
        width_attr = getattr(w, 'width', None)
        if width_attr is not None:
            return width_attr
        rin = getattr(w, '_inner_radius', None)
        r = getattr(w, 'r', None)
        if rin is not None and r is not None:
            return r - rin
        return default_width

    

    def _format_with_first_nonzero(x: float, base_decimals: int, max_decimals: int = 6) -> str:
        """
        Format a positive number with:
          - base_decimals decimals by default
          - BUT if that would round to 0 while x>0, increase decimals until the
            first non-zero digit is shown (up to max_decimals).

        Examples (base_decimals=1):
          8.423  -> "8.4"
          0.002  -> "0.002"
          0.0004 -> "0.0004"  (until max_decimals allows)
        """
        if not np.isfinite(x):
            return "0"
        if x == 0:
            return "0"

        sign = "-" if x < 0 else ""
        x_abs = abs(x)

        d = max(0, int(base_decimals))
        s = f"{x_abs:.{d}f}"
        if float(s) != 0.0:
            # trim trailing zeros/dot for readability (optional)
            if d > 0:
                s = s.rstrip("0").rstrip(".")
            return sign + s

        # If base precision rounds to 0 but x>0, increase decimals until non-zero appears
        for d2 in range(d + 1, int(max_decimals) + 1):
            s2 = f"{x_abs:.{d2}f}"
            if float(s2) != 0.0:
                s2 = s2.rstrip("0").rstrip(".")
                return sign + s2

        return sign + f"{x_abs:.{base_decimals}e}"


    def _two_line_label(val, pct, unit):
        if unit.strip() == '#' and value_decimals == 0:
            val_str = f"{int(round(val))}"
        else:
            val_fmt = f"{{:.{value_decimals}f}}"
            val_str = val_fmt.format(val)
            if value_decimals > 0:
                val_str = val_str.rstrip('0').rstrip('.')

        if not show_percent:
            return f"{val_str}{unit}"

        # ------------------------------------------------------------------
        # adaptive percent formatting
        # If pct is smaller than the resolution implied by percent_decimals,
        # increase decimals so the first non-zero digit is shown.
        # Example:
        #   percent_decimals = 1
        #   44.123  -> 44.1
        #   0.00123 -> 0.001
        # ------------------------------------------------------------------
        if pct == 0 or not np.isfinite(pct):
            pct_str = "0"
        else:
            # base decimals requested by user
            dec = percent_decimals

            # if value is too small, find decimals needed for first non-zero digit
            if pct < 10 ** (-percent_decimals):
                dec = max(
                    percent_decimals,
                    int(np.floor(-np.log10(abs(pct))))
                )

        pct_str = _format_with_first_nonzero(pct, base_decimals=percent_decimals, max_decimals=max(percent_decimals, 6))

        # ------------------------------------------------------------------

        return f"{val_str}{unit}\n({pct_str}\%)"

    
    
    def _annotate_wedges(
        ax, wedges, vals, pcts, unit,
        place_outside=False, small_thresh=None,
        ring_outer_radius=None, ring_width_default=None,
        inside_radius=None,
        for_outer=False, small_outer_separation=0.20
    ):
        small_thresh = small_thresh or 0.0

        for idx, (w, v, p) in enumerate(zip(wedges, vals, pcts)):
            if not np.isfinite(v) or v <= 0:
                continue

            txt = _two_line_label(v, p, unit)

            ang = (w.theta2 + w.theta1) / 2.0
            ang_rad = np.deg2rad(ang)
            r0 = _wedge_outer_radius(w, ring_outer_radius if ring_outer_radius is not None else 1.0)
            width = _wedge_width(w, ring_width_default if ring_width_default is not None else 0.3)
            
            
            r_mid = r0 - width / 6 # use 2.0 for middle
            r_text = inside_radius if inside_radius is not None else r_mid
            x = r_text * np.cos(ang_rad)
            y = r_text * np.sin(ang_rad)

            if place_outside and v < small_thresh:
                r_out = r0 + 0.06
                xo = r_out * np.cos(ang_rad)
                yo = r_out * np.sin(ang_rad)

                if for_outer:
                    if idx == 1:   # MV
                        xo += small_outer_separation
                    elif idx == 2:  # HV
                        xo -= small_outer_separation

                ax.annotate(
                    txt, xy=(x + 0.02, y), xytext=(xo, yo), # Control location of arrow between label and outer ring
                    textcoords='data', ha='center', va='center',
                    arrowprops=dict(arrowstyle='-', lw=0.6, color='0.3'),
                    **font_labels
                )
            else:
                if (not place_outside) and v < small_thresh:
                    ax.annotate(
                        txt, xy=(x, y), xytext=(x * 0.85, y * 0.85),
                        textcoords='data', ha='center', va='center',
                        arrowprops=dict(arrowstyle='-', lw=0.6, color='0.3'),
                        **font_labels
                    )
                else:
                    ax.text(x, y, txt, ha='center', va='center', **font_labels)

    # ----------------------------------------------------------------------
    # Map helpers
    # ----------------------------------------------------------------------
    PERIOD_SUMMARY = level1_key 
    
    def _get_city_region_df(city, region):
        return transformers_summary_dict[PERIOD_SUMMARY][(smart_ds_year, city, region)]

    def _add_scale_bar(ax, length_km=SCALE_BAR_KM, x_frac=0.08, y_frac=0.07):
        """Add a simple scale bar to a map whose CRS units are metres."""
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        x_range = xmax - xmin
        y_range = ymax - ymin
        length_m = float(length_km) * 1000.0

        # Position near the lower-left corner, while keeping the full bar inside the map.
        x0 = xmin + x_frac * x_range
        x0 = min(x0, xmax - length_m - 0.05 * x_range)
        y0 = ymin + y_frac * y_range
        x1 = x0 + length_m

        tick_h = 0.012 * y_range

        ax.plot([x0, x1], [y0, y0], color='white', lw=5.0,
                solid_capstyle='butt', zorder=100)
        ax.plot([x0, x0], [y0 - tick_h, y0 + tick_h], color='white', lw=5.0,
                solid_capstyle='butt', zorder=100)
        ax.plot([x1, x1], [y0 - tick_h, y0 + tick_h], color='white', lw=5.0,
                solid_capstyle='butt', zorder=100)

        ax.plot([x0, x1], [y0, y0], color='black', lw=2.0,
                solid_capstyle='butt', zorder=101)
        ax.plot([x0, x0], [y0 - tick_h, y0 + tick_h], color='black', lw=2.0,
                solid_capstyle='butt', zorder=101)
        ax.plot([x1, x1], [y0 - tick_h, y0 + tick_h], color='black', lw=2.0,
                solid_capstyle='butt', zorder=101)

        ax.text(
            (x0 + x1) / 2.0,
            y0 + 2.0 * tick_h,
            f'{length_km:g} km',
            ha='center', va='bottom',  color='black', zorder=102,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=0.8)
        )

    # collect voltages across all city/regions so legend is globally consistent
    all_sec_v = []
    for city in cities:
        regs = city_regions_to_run[city]
        for region in regs:
            try:
                df = _get_city_region_df(city, region)
                sec = df['KV rating [kV]'].apply(
                    lambda x: (x[1] if isinstance(x, (list, tuple)) and len(x) > 1 else np.nan)
                )
                all_sec_v.append(sec.to_numpy(dtype=float))
            except Exception:
                continue
    all_sec_v = np.concatenate([v[np.isfinite(v)] for v in all_sec_v if v is not None])

    if voltage_order is None:
        unique_voltages = np.unique(all_sec_v)
        unique_voltages = np.sort(unique_voltages)
    else:
        present = set(all_sec_v.tolist())
        unique_voltages = [v for v in voltage_order if v in present]

    if len(unique_voltages) != 4:
        raise ValueError(
            f"Expected 4 unique secondary voltage values, got {len(unique_voltages)}: {unique_voltages}"
        )

    if isinstance(voltage_colors, dict):
        try:
            color_list = [mcolors.to_rgba(voltage_colors[v], alpha=fill_alpha) for v in unique_voltages]
        except KeyError as e:
            raise ValueError(f"Missing color for voltage {e.args[0]} in voltage_colors dict.")
    elif isinstance(voltage_colors, (list, tuple)):
        if len(voltage_colors) != 4:
            raise ValueError("voltage_colors list must have exactly 4 HEX colors.")
        color_list = [mcolors.to_rgba(c, alpha=fill_alpha) for c in voltage_colors]
    else:
        # default G-Y-O-R palette
        default_hex = ['#81FB48', '#FCDF90', '#F7A35C', '#EA848A']
        color_list = [mcolors.to_rgba(c, alpha=fill_alpha) for c in default_hex]

    cmap = ListedColormap(color_list)
    v_to_idx = {v: i for i, v in enumerate(unique_voltages)}

    # ======================================================================
    # Create 4x3 figure
    # ======================================================================
    fig, axes = plt.subplots(4, 3, figsize=figsize)
    axes = np.atleast_2d(axes)

    if suptitle:
        fig.suptitle(suptitle,  y=0.98)

    # Legend for LV/MV/HV colors (donuts) at top center
    legend_handles_donut = [

        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=donut_colors[0],
            markeredgecolor='none',
            markeredgewidth=0,
            markersize=10,
            label='Low-voltage'
        ),
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=donut_colors[1],
            markeredgecolor='none',
            markeredgewidth=0,
            markersize=10,
            label='Medium-voltage'
        ),
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=donut_colors[2],
            markeredgecolor='none',
            markeredgewidth=0,
            markersize=10,
            label='High-voltage'
        ),
        
        # outline-only circle
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor='none',
            markeredgecolor='black',
            markeredgewidth=1.0,
            markersize=10,
            label='Shift to at-risk or critical loading'
        ),
    ]
      
    
    
    legend_kwargs = {}
    if font_legend is not None:
        legend_kwargs["prop"] = font_legend    
    
    fig.legend(
        handles=legend_handles_donut,
        loc='upper center',
        ncol=4,
        bbox_to_anchor=(0.5, 1.05),
        frameon=False,
        **legend_kwargs
    )

    
    
    # ======================================================================
    # Rows 1--3: maps (a--i) — one map per city-region combination
    # ======================================================================
    for city in cities:
        if city not in city_regions_to_run:
            raise KeyError(f"Missing city '{city}' in city_regions_to_run.")
        if len(city_regions_to_run[city]) != 3:
            raise ValueError(
                f"plot_donuts_and_maps_4x3_v1 expects exactly 3 regions for each city. "
                f"City '{city}' has {len(city_regions_to_run[city])}: "
                f"{city_regions_to_run[city]}"
            )

    for col_idx, city in enumerate(cities):
        regs = city_regions_to_run[city]

        for row_idx, region in enumerate(regs):
            ax = axes[row_idx, col_idx]
            ax.set_aspect('equal')

            try:
                df_region = _get_city_region_df(city, region).copy()
            except Exception as e:
                raise KeyError(
                    f"Missing map data for {PERIOD_SUMMARY} / "
                    f"({smart_ds_year}, {city}, {region}): {e}"
                )

            # marker size bins 
            size_bins = [
                (0, 1e2), (1e2, 1e3), (1e3, 1e4),
                (1e4, 1e5), (1e5, 1e6), (1e6, 1e7), (1e7, np.inf)
            ]
            size_values = [5, 10, 20, 80, 100, 150, 300]
            df_region['Marker Size'] = _assign_marker_size(
                df_region, size_bins, size_values
            )

            # secondary voltage & class
            df_region['Secondary kV'] = df_region['KV rating [kV]'].apply(
                lambda x: (
                    x[1]
                    if isinstance(x, (list, tuple)) and len(x) > 1
                    else np.nan
                )
            ).astype(float)
            df_region['Voltage Class'] = df_region['Secondary kV'].map(v_to_idx)

            # newly overloaded = candidate or critical
            df_region['Newly Overloaded'] = (
                ((df_region[hist_col] < baseline_overload_thr) &
                 (df_region[fut_col] >= future_overload_thr)) |
                ((df_region[hist_col] < baseline_overload_thr_critical) &
                 (df_region[fut_col] >= future_overload_thr_critical))
            )

            # Project to a local UTM CRS so map distances are in metres.
            if city not in CITY_UTM:
                raise ValueError(
                    f"No local UTM CRS is defined for city '{city}'. "
                    f"Add it to CITY_UTM inside plot_donuts_and_maps_4x3_v1()."
                )
            map_crs = CITY_UTM[city]

            gdf = gpd.GeoDataFrame(
                df_region,
                geometry=gpd.points_from_xy(
                    df_region['Long'], df_region['Lat']
                ),
                crs='EPSG:4326'
            ).to_crs(map_crs)

            # --------------------------------------------------------------
            #  colored transformer markers
            # --------------------------------------------------------------

            for voltage_idx, voltage in enumerate(unique_voltages):
                gdf_v = gdf[gdf['Secondary kV'] == voltage].copy()
                if gdf_v.empty:
                    continue

                # Larger markers first, so smaller coincident markers can remain visible.
                gdf_v = gdf_v.sort_values('Marker Size', ascending=False)

                gdf_v.plot(
                    ax=ax,
                    color=color_list[voltage_idx],
                    markersize=gdf_v['Marker Size'],
                    marker='o',
                    edgecolor='none',
                    zorder=2 + voltage_idx
                )

            # --------------------------------------------------------------
            #  newly-overloaded outlines
            # --------------------------------------------------------------

            for voltage_idx, voltage in enumerate(unique_voltages):
                gdf_v_over = gdf[
                    (gdf['Secondary kV'] == voltage) &
                    (gdf['Newly Overloaded'])
                ].copy()

                if gdf_v_over.empty:
                    continue

                gdf_v_over = gdf_v_over.sort_values(
                    'Marker Size', ascending=False
                )

                if voltage < 1:
                    this_outline_width = outline_width_LV
                elif voltage <= 35:
                    this_outline_width = outline_width_MV
                else:
                    this_outline_width = outline_width_HV

                gdf_v_over.plot(
                    ax=ax,
                    color='none',
                    edgecolor=outline_color,
                    markersize=gdf_v_over['Marker Size'] * 1.2,
                    linewidth=this_outline_width,
                    marker='o',
                    zorder=50 + voltage_idx
                )

            # Contextily warps the basemap into the city's local UTM projection.
            cx.add_basemap(
                ax,
                source=cx.providers.CartoDB.Positron,
                crs=map_crs
            )
            for text in ax.texts:
                # shrink basemap attribution
                text.set_color('gray')
                text.set_alpha(0.5)

            # 10 km scale bar, using metre-based UTM coordinates.
            _add_scale_bar(ax)

            ax.set_axis_off()

            # City name appears once at the top of each column.
            if row_idx == 0:
                ax.set_title(
                    f"{_get_city_label(city)}",
                    pad=2
                )

            # Small region identifier within each map panel.
            ax.text(
                0.98, 0.98,
                f"Region {row_idx + 1} ({region})",
                transform=ax.transAxes,
                ha='right', va='top',
                zorder=110,
                bbox=dict(
                    facecolor='white',
                    edgecolor='none',
                    alpha=0.80,
                    pad=1.5
                )
            )

    # ======================================================================
    # Row 4: donuts (j, k, l)
    # ======================================================================
    # Store the same numerical values used to draw the donut charts.
    donut_records = []

    for ax, city in zip(axes[3, :], cities):
        ax.set_aspect('equal')

        df = _get_city_donut_df(city)
        outer_vals, outer_pcts = _values_per_rows(df, outer_rows, outer_value_col, outer_percent_col)
        inner_vals, inner_pcts = _values_per_rows(df, inner_rows, inner_value_col, inner_percent_col)

        outer_vals = np.where(np.isfinite(outer_vals), outer_vals, 0.0)
        inner_vals = np.where(np.isfinite(inner_vals), inner_vals, 0.0)
        outer_vals = np.where(outer_vals < 0, 0.0, outer_vals)
        inner_vals = np.where(inner_vals < 0, 0.0, inner_vals)
        outer_pcts = np.where(np.isfinite(outer_pcts), outer_pcts, 0.0)
        inner_pcts = np.where(np.isfinite(inner_pcts), inner_pcts, 0.0)

        # Save exactly the values represented in the donut rings.
        for voltage_level, number, number_pct, capacity, capacity_pct in zip(
            ('LV', 'MV', 'HV'),
            outer_vals, outer_pcts, inner_vals, inner_pcts
        ):
            donut_records.append({
                'City': _get_city_label(city),
                'Voltage level': voltage_level,
                'Number': number,
                'Number [%]': number_pct,
                'Capacity [MVA]': capacity,
                'Capacity [%]': capacity_pct,
            })

        # city label
        ax.text(
            0, outer_radius + city_label_offset,
            _get_city_label(city),
            ha='center', va='center', **font_title_city
        )

        # outer ring
        if np.allclose(outer_vals.sum(), 0):
            outer_vals_plot = np.array([1, 1, 1], float)
            colors_outer = ['#dddddd'] * 3
        else:
            outer_vals_plot = outer_vals.astype(float).copy()
            if small_outer_slice_scale != 1.0 and small_thresh_outer is not None:
                small_mask = (outer_vals_plot > 0) & (outer_vals_plot < small_thresh_outer)
                outer_vals_plot[small_mask] *= float(small_outer_slice_scale)
            colors_outer = donut_colors

        wedges_outer, _ = ax.pie(
            outer_vals_plot,
            radius=outer_radius,
            colors=colors_outer,
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=outer_width, edgecolor='white')
        )

        # outer title (bottom-left of the donut panel, in axes coordinates)
        ax.text(
            0.04, 0.08,                      # x,y in axes fraction (0..1)
            outer_title,
            transform=ax.transAxes,          # anchor to the axes box
            ha='left', va='bottom',
            **font_ring_titles
        )



        _annotate_wedges(
            ax, wedges_outer, outer_vals, outer_pcts, outer_unit,
            place_outside=True, small_thresh=small_thresh_outer,
            ring_outer_radius=outer_radius, ring_width_default=outer_width,
            for_outer=True, small_outer_separation=small_outer_separation
        )

        # inner ring
        if np.allclose(inner_vals.sum(), 0):
            inner_vals_plot = np.array([1, 1, 1], float)
            colors_inner = ['#eeeeee'] * 3
        else:
            inner_vals_plot = inner_vals
            colors_inner = donut_colors

        wedges_inner, _ = ax.pie(
            inner_vals_plot,
            radius=inner_radius,
            colors=colors_inner,
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=inner_width, edgecolor='white')
        )

        ax.text(0, 0, inner_title, ha='center', va='center', **font_ring_titles)

        inner_text_radius = inner_radius - inner_width * 0.4
        _annotate_wedges(
            ax, wedges_inner, inner_vals, inner_pcts, inner_unit,
            place_outside=False, small_thresh=small_thresh_inner,
            ring_outer_radius=inner_radius, ring_width_default=inner_width,
            inside_radius=inner_text_radius
        )

        ax.set_axis_off()


    # ======================================================================
    # Panel labels (a)-(l)
    # ======================================================================
    panel_labels = list('abcdefghijkl')
    k = 0
    for i in range(4):
        for j in range(3):
            ax = axes[i, j]
            ax.text(
                0.01, 0.99, panel_labels[k],
                transform=ax.transAxes,
                ha='left', va='top',
                fontweight='bold',
                zorder=120
            )
            k += 1

    plt.tight_layout()
    if savepath:
        plt.savefig(
            savepath,
            bbox_inches='tight',
            pad_inches=0.5,
            dpi=600
        )
    plt.show()

    donut_df = pd.DataFrame(donut_records)
    return donut_df

## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

smart_ds_year = config['smart_ds_years'][0]

## Initialize parameters for saving paths
output_pf_path = config['output_pf_path']

with open(config_path, "r") as file:
    pf_config = yaml.safe_load(file)
    
solution_mode = pf_config['solution_mode']

# --- put rows below in the load config cell ---
start_row_percent = config['start_row_percent']
top_percent_mdh = config['top_percent_mdh']

# File names
transformers_file_name = f"transformers_top_{start_row_percent}_{top_percent_mdh}_percent"
lines_file_name = f"lines_top_{start_row_percent}_{top_percent_mdh}_percent"

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(f"solar_share: {solar_share}\n battery_share: {battery_share}\n solar_battery_scenario_folder: {solar_battery_scenario_folder}")


print(f"\nsolution_mode: {solution_mode} \noutput_pf_path:{output_pf_path}")

print(f"\nstart_row_percent: {start_row_percent} \ntop_percent_mdh: {top_percent_mdh}")

## Set parameters

In [ ]:
save_folder = "all_regions" 

TGW_scenario = "rcp45hotter"

TGW_weather_year = '2030_2059'

fig_name_3_reg = f"figures/maps/overloaded_xfers_by_kv/donuts_and_three_regions_GYR_v1_{TGW_weather_year}_{TGW_scenario}_{save_folder}.pdf"
fig_name_9_reg = f"figures/maps/overloaded_xfers_by_kv/donuts_and_nine_regions_GYR_v1_{TGW_weather_year}_{TGW_scenario}_{save_folder}.pdf"

# Create the directory structure if it doesn't exist
os.makedirs(os.path.dirname(fig_name_9_reg), exist_ok=True)

# Grid reinforcements Thresholds
xfm_cand = 80   # transformers candidate overloading threshold
xfm_crit = 100  # transformers critical overloading threshold
line_cand = 67  # lines candidate overloading threshold
line_crit = 100 # lines critical overloading threshold


LV_MV_threshold = 1 # [kV]
MV_HV_threshold = 35 # [kV]

# Loading columns
hist_col = "median_annual_max_loading_historical_1990_2019"      # baseline loading column
fut_col  = "median_annual_max_loading_rcp45hotter_2030_2059"     # future loading column

# List of cities to process
cities = ["AUS", "GSO", "SFO"]

# Select regions to process and show in donut charts
CITY_REGIONS_TO_RUN = {
    "GSO": ["rural", "industrial", "urban-suburban"],
    "SFO": ["P1U", "P2U", "P1R"],
    "AUS": ["P1U", "P1R", "P2U"],
}

# Select representitive regions to show on the map  
CITY_REGIONS_TO_MAP = {
    "AUS": ["P1U"],
    "GSO": ["urban-suburban"],
    "SFO": ["P2U"],
}

## Parameters for figures format
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]
mpl.rcParams['text.usetex'] = True

## Load merged dictionary w/ summary statistics across years

In [ ]:
# ============================================================
# Load period-level summary dictionaries
# ============================================================
summary_save_dir = os.path.join(
    output_pf_path,
    save_folder,
    "summary_across_weather_years",
    TGW_scenario,
    solar_battery_scenario_folder,
)
transformers_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{transformers_file_name}_summary_across_weather_years.joblib",
)

lines_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{lines_file_name}_summary_across_weather_years.joblib",
)

# Check files exist before loading
for path in [
    transformers_summary_across_years_path,
    lines_summary_across_years_path,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

transformers_dict_summary_multi_region = joblib.load(
    transformers_summary_across_years_path
)

lines_dict_summary_multi_region = joblib.load(
    lines_summary_across_years_path
)

print("Loaded:")
print(transformers_summary_across_years_path)
print(lines_summary_across_years_path)

# ============================================================
# Inspect loaded dictionaries
# ============================================================

print("\nTransformer summary dictionary nested keys:")
file_ops.print_nested_keys_structure(transformers_dict_summary_multi_region)

print("\nTransformer summary dictionary sample dataframe:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(transformers_dict_summary_multi_region)

## Process data - Concatenate regions to city level, add voltage and MVA

In [ ]:
# Add voltage
for city, regions in CITY_REGIONS_TO_RUN.items():
    for region in regions:
        if 'secondary KV rating [kV]' not in transformers_dict_summary_multi_region[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city,region)].columns:
            transformers_dict_summary_multi_region[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city,region)].insert(7, 'secondary KV rating [kV]', secondary_kv_series_transformers(transformers_dict_summary_multi_region[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city,region)]), allow_duplicates=False)

### Concatenate regions to single city level dataframes (e.g., a single df with all transformers in GSO)
transformers_dict_summary_multi_region_agg_by_city, lines_dict_summary_multi_region_agg_by_city = (
    df_ops.concat_regions_to_city(
        transformers_dict_summary_multi_region,
        lines_dict_summary_multi_region,
        TGW_weather_year,
        TGW_scenario,
        smart_ds_year,
        CITY_REGIONS_TO_RUN,
    )
)


cities = ["AUS","GSO","SFO"]
for city in cities:
    if 'MVA rating [MVA]' not in transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].columns:
        transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].insert(8, "MVA rating [MVA]",  mva_series_transformers(transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]), allow_duplicates=False)
    if 'MVA rating [MVA]' not in lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].columns:
        lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].insert(9, "MVA rating [MVA]",  mva_series_lines(lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]), allow_duplicates=False)
display(transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].head(2))

## Create table of overloaded (B_and_F,B,F,F_minus_B, F_not_B)

In [ ]:
# Initialize transformers
lines_overloaded = {}
lines_overloaded[(TGW_weather_year, TGW_scenario)] = {}
transformers_overloaded = {}
transformers_overloaded[(TGW_weather_year, TGW_scenario)] = {}


## --- Create an overloaded data table for transformers --- 
for city in cities:
    df = transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
    
    # Define masks
    mask_cand_B = in_threshold_range(df[hist_col], xfm_cand, xfm_crit)
    mask_cand_F = in_threshold_range(df[fut_col], xfm_cand, xfm_crit)
    mask_crit_B = above_threshold(df[hist_col], xfm_crit)
    mask_crit_F = above_threshold(df[fut_col], xfm_crit)
    mask_B = mask_cand_B | mask_crit_B
    mask_F = mask_cand_F | mask_crit_F
    mask_cand_F_not_B = (
    in_threshold_range(df[fut_col], xfm_cand, xfm_crit) &
    below_threshold(df[hist_col], xfm_cand))
    mask_crit_F_not_B = (
    above_threshold(df[fut_col], xfm_crit) &
    below_threshold(df[hist_col], xfm_cand))
    mask_cand_B_to_crit_F = mask_cand_B & mask_crit_F
       
    xfer_mask_LV = df['KV rating [kV]'].str[-1] < LV_MV_threshold
    xfer_mask_MV = (df['KV rating [kV]'].str[-1] > LV_MV_threshold) & (df['KV rating [kV]'].str[-1] <= MV_HV_threshold)
    xfer_mask_HV = df['KV rating [kV]'].str[-1] > MV_HV_threshold


    rows = []
    # --- TOTAL COUNTS ---
    # --- all ---
    rows.append({
        "Metric": "Cand+Crit all #",
        "B": mask_B.sum(),
        "F": mask_F.sum(),
        "F_minus_B": mask_F.sum() - mask_B.sum(),
        "F_risk_escalation": risk_escalation_count(True),  # no voltage mask
    })
    # --- LV ---
    rows.append({
        "Metric": "Cand+Crit LV #",
        "B": (xfer_mask_LV & mask_B).sum(),
        "F": (xfer_mask_LV & mask_F).sum(),
        "F_minus_B": (xfer_mask_LV & mask_F).sum() - (xfer_mask_LV & mask_B).sum(),
        "F_risk_escalation": risk_escalation_count(xfer_mask_LV),
    })
    # --- MV ---
    rows.append({
        "Metric": "Cand+Crit MV #",
        "B": (xfer_mask_MV & mask_B).sum(),
        "F": (xfer_mask_MV & mask_F).sum(),
        "F_minus_B": (xfer_mask_MV & mask_F).sum() - (xfer_mask_MV & mask_B).sum(),
        "F_risk_escalation": risk_escalation_count(xfer_mask_MV),
    })
    # --- HV ---
    rows.append({
        "Metric": "Cand+Crit HV #",
        "B": (xfer_mask_HV & mask_B).sum(),
        "F": (xfer_mask_HV & mask_F).sum(),
        "F_minus_B": (xfer_mask_HV & mask_F).sum() - (xfer_mask_HV & mask_B).sum(),
        "F_risk_escalation": risk_escalation_count(xfer_mask_HV),
    })


    # --- MVA SUMS ---
    # --- ALL ---
    rows.append({
        "Metric": "Cand+Crit all MVA",
        "B": df.loc[mask_B, 'MVA rating [MVA]'].sum(),
        "F": df.loc[mask_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B": (
            df.loc[mask_F, 'MVA rating [MVA]'].sum()
            - df.loc[mask_B, 'MVA rating [MVA]'].sum()
        ),
        "F_risk_escalation": risk_escalation_mva(True),
    })

    # --- LV ---
    rows.append({
        "Metric": "Cand+Crit LV MVA",
        "B": df.loc[xfer_mask_LV & mask_B, 'MVA rating [MVA]'].sum(),
        "F": df.loc[xfer_mask_LV & mask_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B": (
            df.loc[xfer_mask_LV & mask_F, 'MVA rating [MVA]'].sum()
            - df.loc[xfer_mask_LV & mask_B, 'MVA rating [MVA]'].sum()
        ),
        "F_risk_escalation": risk_escalation_mva(xfer_mask_LV),
    })

    # --- MV ---
    rows.append({
        "Metric": "Cand+Crit MV MVA",
        "B": df.loc[xfer_mask_MV & mask_B, 'MVA rating [MVA]'].sum(),
        "F": df.loc[xfer_mask_MV & mask_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B": (
            df.loc[xfer_mask_MV & mask_F, 'MVA rating [MVA]'].sum()
            - df.loc[xfer_mask_MV & mask_B, 'MVA rating [MVA]'].sum()
        ),
        "F_risk_escalation": risk_escalation_mva(xfer_mask_MV),
    })

    # --- HV ---
    rows.append({
        "Metric": "Cand+Crit HV MVA",
        "B": df.loc[xfer_mask_HV & mask_B, 'MVA rating [MVA]'].sum(),
        "F": df.loc[xfer_mask_HV & mask_F, 'MVA rating [MVA]'].sum(),
        "F_minus_B": (
            df.loc[xfer_mask_HV & mask_F, 'MVA rating [MVA]'].sum()
            - df.loc[xfer_mask_HV & mask_B, 'MVA rating [MVA]'].sum()
        ),
        "F_risk_escalation": risk_escalation_mva(xfer_mask_HV),
    })
    df_rows = pd.DataFrame(rows)
    # Round all numeric columns to 2 decimal places (in place)
    numeric = df_rows.select_dtypes(include="number").columns
    df_rows[numeric] = df_rows[numeric].round(2)
    transformers_overloaded[(TGW_weather_year, TGW_scenario)][city] = df_rows
    
city = 'AUS'
display(transformers_overloaded[(TGW_weather_year, TGW_scenario)][city])

## Add percentage values

In [ ]:
for city in cities:
    df_data = transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
    df_overloaded = transformers_overloaded[(TGW_weather_year, TGW_scenario)][city]
    
    xfer_mask_LV = df_data['KV rating [kV]'].str[-1] < LV_MV_threshold
    xfer_mask_MV = (df_data['KV rating [kV]'].str[-1] > LV_MV_threshold) & (df_data['KV rating [kV]'].str[-1] <= MV_HV_threshold)
    xfer_mask_HV = df_data['KV rating [kV]'].str[-1] > MV_HV_threshold
    
    # Denominators (counts for rows 1-4)
    denom_all_cnt = len(df_data)
    denom_lv_cnt  = xfer_mask_LV.sum()
    denom_mv_cnt  = xfer_mask_MV.sum()
    denom_hv_cnt  = xfer_mask_HV.sum()

    # Denominators (MVA sums for rows 5-8)
    denom_all_mva = df_data['MVA rating [MVA]'].sum()
    denom_lv_mva  = df_data.loc[xfer_mask_LV, 'MVA rating [MVA]'].sum()
    denom_mv_mva  = df_data.loc[xfer_mask_MV, 'MVA rating [MVA]'].sum()
    denom_hv_mva  = df_data.loc[xfer_mask_HV, 'MVA rating [MVA]'].sum()

    ## Build list of denominators aligned to row order in df_overloaded
   
    # Add percentage of LV/MV/HV out of overall LV+MV+HV 
    totals_by_row = [
        denom_all_cnt,  # row 1: Cand+Crit all #
        denom_all_cnt,   # row 2: Cand+Crit LV #
        denom_all_cnt,   # row 3: Cand+Crit MV #
        denom_all_cnt,   # row 4: Cand+Crit HV #
        denom_all_mva,  # row 5: Cand+Crit all MVA
        denom_all_mva,   # row 6: Cand+Crit LV MVA
        denom_all_mva,   # row 7: Cand+Crit MV MVA
        denom_all_mva,   # row 8: Cand+Crit HV MVA
    ]
    
    transformers_overloaded[(TGW_weather_year, TGW_scenario)][city] = add_percentage_columns_by_row_totals(
        df_overloaded,
        totals_by_row=totals_by_row,
        start_col=1,       # B starts at col 1
        n_numeric=4,       # B, F, F_minus_B, F_not_B
        suffix="_pct_of_all",
        decimals=3,
    )

## Overloaded Maps and LV/MV/HV donut pies

In [ ]:
start_time = time.time()

fontsize = 18
plt.rcParams.update({
    "font.size": fontsize,               
    "axes.labelsize": fontsize,
    "axes.titlesize": fontsize,          
    "figure.titlesize": fontsize,        
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "legend.fontsize": fontsize,
    "legend.title_fontsize": fontsize,
})

donut_df = plot_donuts_and_maps_2x3_v3(
    transformers_overloaded=transformers_overloaded,
    transformers_summary_dict=transformers_dict_summary_multi_region,
    city_regions_to_run=CITY_REGIONS_TO_MAP,
    smart_ds_year = smart_ds_year,
    level1_key=(TGW_weather_year, TGW_scenario),
    cities=cities,
    outer_rows=('Cand+Crit LV #','Cand+Crit MV #','Cand+Crit HV #'),
    inner_rows=('Cand+Crit LV MVA','Cand+Crit MV MVA','Cand+Crit HV MVA'),
    outer_value_col='F_risk_escalation',
    outer_percent_col='F_risk_escalation_pct_of_all',
    inner_value_col='F_risk_escalation',
    inner_percent_col='F_risk_escalation_pct_of_all',
    donut_colors=('#50ad9f','#e9c716','#E34A33'),
    outer_radius=0.8, outer_width=0.2,
    inner_radius=0.62, inner_width=0.2,
    small_thresh_outer=80,
    small_thresh_inner=10,
    outer_unit='', inner_unit=' MVA',
    font_title_city=None,
    font_ring_titles=None,
    font_labels={'size': fontsize - 6},
    font_legend=None,
    outer_title='Number \nof transformers', 
    inner_title='Capacity', 
    outer_title_offset=0.21,
    city_label_offset=0.38,
    small_outer_separation=0.6,
    small_outer_slice_scale=7,
    value_decimals=0,
    percent_decimals=1,
    show_percent=True,
    hist_col=hist_col,
    fut_col=fut_col,
    baseline_overload_thr=80,
    future_overload_thr=80,
    baseline_overload_thr_critical=100,
    future_overload_thr_critical=100,
    voltage_order=[0.12, 0.48, 12.47, 69.0],
    voltage_colors={0.12:'#50ad9f', 0.48:'#50ad9f', 12.47:'#e9c716', 69.0:'#E34A33'},
    outline_color='#000000',
    outline_width_LV=0.5,
    outline_width_MV=0.7,
    outline_width_HV=0.7,
    figsize=(14, 8),
    suptitle=None, 
    savepath = fig_name_3_reg
)

donut_table = donut_df.copy()

donut_table['Number'] = donut_table['Number'].round(0).astype(int)
donut_table['Number [%]'] = donut_table['Number [%]'].round(1)
donut_table['Capacity [MVA]'] = donut_table['Capacity [MVA]'].round(1)
donut_table['Capacity [%]'] = donut_table['Capacity [%]'].round(1)

latex_table = donut_table.to_latex(
    index=False,
    escape=True,
    caption=(
        'Climate-driven transformer overloading by voltage level. '
        'Values correspond to those shown in the donut charts in Fig.~3.'
    ),
    label='tab:transformer_overloading_by_voltage',
    column_format='llrrrr',
)

print(latex_table)

end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")

## Table of percentage of affected classes (risk escalation)

In [ ]:
# ------------------------------------------------------------
# Risk escalation by voltage class:
#   1) % of transformers within that voltage class
#   2) % of all transformers in the city
# ------------------------------------------------------------

import pandas as pd

# Total number of transformers by city and voltage class
transformer_totals = {
    "Austin": {
        "LV": 42658,
        "MV": 81,
        "HV": 11,
    },
    "Greensboro": {
        "LV": 25929,
        "MV": 46,
        "HV": 7,
    },
    "San Francisco": {
        "LV": 35455,
        "MV": 110,
        "HV": 19,
    },
}

# Start from the existing donut table
risk_table = donut_df.copy()

# Add total number of transformers in each voltage class
risk_table["Total in voltage class"] = risk_table.apply(
    lambda row: transformer_totals[row["City"]][row["Voltage level"]],
    axis=1,
)

# Add total number of transformers in each city
city_totals = {
    city: sum(levels.values())
    for city, levels in transformer_totals.items()
}

risk_table["Total transformers"] = risk_table["City"].map(city_totals)

# Calculate percentages
risk_table["Of voltage class [%]"] = (
    100
    * risk_table["Number"]
    / risk_table["Total in voltage class"]
)

risk_table["Of all transformers [%]"] = (
    100
    * risk_table["Number"]
    / risk_table["Total transformers"]
)

# Keep only relevant columns
risk_table = risk_table[
    [
        "City",
        "Voltage level",
        "Number",
        "Total in voltage class",
        "Of voltage class [%]",
        "Of all transformers [%]",
    ]
]

display(risk_table)

latex_table = risk_table.to_latex(
    index=False,
    caption=(
        "caption"
    ),
    label="tab:transformer_risk_escalation_by_voltage",
    column_format="llrrrr",
    formatters={
        "Number": lambda x: f"{int(x)}",
        "Total in voltage class": lambda x: f"{int(x)}",
        "Of voltage class [%]": lambda x: f"{x:.2f}",
        "Of all transformers [%]": lambda x: f"{x:.3f}",
    },
    escape=False,
)

print(latex_table)

## Plot all regions

In [ ]:
CITY_REGIONS_TO_MAP_all_regions = CITY_REGIONS_TO_RUN
    

fontsize = 16
plt.rcParams.update({
    "font.size": fontsize,                
    "axes.labelsize": fontsize,
    "axes.titlesize": fontsize,           
    "figure.titlesize": fontsize,       
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "legend.fontsize": fontsize,
    "legend.title_fontsize": fontsize,
})

start_time = time.time()

donut_df = plot_donuts_and_maps_4x3_v1(
    transformers_overloaded=transformers_overloaded,
    transformers_summary_dict=transformers_dict_summary_multi_region,
    city_regions_to_run=CITY_REGIONS_TO_MAP_all_regions,
    smart_ds_year = smart_ds_year,
    level1_key=(TGW_weather_year, TGW_scenario),
    cities=cities,
    outer_rows=('Cand+Crit LV #','Cand+Crit MV #','Cand+Crit HV #'),
    inner_rows=('Cand+Crit LV MVA','Cand+Crit MV MVA','Cand+Crit HV MVA'),
    outer_value_col='F_risk_escalation',
    outer_percent_col='F_risk_escalation_pct_of_all',
    inner_value_col='F_risk_escalation',
    inner_percent_col='F_risk_escalation_pct_of_all',
    donut_colors=('#50ad9f','#e9c716','#E34A33'),
    outer_radius=0.8, outer_width=0.2,
    inner_radius=0.62, inner_width=0.2,
    small_thresh_outer=80,
    small_thresh_inner=10,
    outer_unit='', inner_unit=' MVA',
    font_title_city=None,
    font_ring_titles=None,
    font_labels={'size': fontsize - 3},
    font_legend=None,    outer_title='Number \nof transformers',
    inner_title='Capacity', 
    outer_title_offset=0.21,
    city_label_offset=0.38,
    small_outer_separation=0.5,
    small_outer_slice_scale=7,
    value_decimals=0,
    percent_decimals=1,
    show_percent=True,
    hist_col=hist_col,
    fut_col=fut_col,
    baseline_overload_thr=80,
    future_overload_thr=80,
    baseline_overload_thr_critical=100,
    future_overload_thr_critical=100,
    voltage_order=[0.12, 0.48, 12.47, 69.0],
    voltage_colors={0.12:'#50ad9f', 0.48:'#50ad9f', 12.47:'#e9c716', 69.0:'#E34A33'},
    outline_color='#000000',
    outline_width_LV=0.5,
    outline_width_MV=0.7,
    outline_width_HV=0.7,
    figsize=(14, 15),
    suptitle=None, 
    savepath = fig_name_9_reg
)

donut_table = donut_df.copy()

donut_table['Number'] = donut_table['Number'].round(0).astype(int)
donut_table['Number [%]'] = donut_table['Number [%]'].round(1)
donut_table['Capacity [MVA]'] = donut_table['Capacity [MVA]'].round(1)
donut_table['Capacity [%]'] = donut_table['Capacity [%]'].round(1)

latex_table = donut_table.to_latex(
    index=False,
    escape=True,
    caption=(
        'caption'
    ),
    label='tab:transformer_overloading_by_voltage',
    column_format='llrrrr',
)

print(latex_table)

end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")